# MedVision AI: train the ISIC 2019 model on Colab

This notebook trains, evaluates and exports the skin-lesion classifier with the
repository's own CLI, so the result matches what CI tests.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*, and keep this tab open while it runs (free sessions stop when idle).
3. Allow Google Drive access when asked: every result is copied to `MyDrive/medvisionai/`.

It downloads about 10 GB (ISIC 2019, CC BY-NC 4.0). Training stops early when
validation loss stops improving; expect a few hours on a T4. Any failing step
stops the notebook, so a broken run never reaches the export.

Some steps print nothing for minutes (installing, importing TensorFlow,
unzipping, training). **Do not press Stop**: a `KeyboardInterrupt` means the
cell was interrupted. To resume, click the cell and use *Runtime → Run cell and below*.

## 1. Helpers and GPU check

In [ ]:
import os
import subprocess


def sh(command: str) -> None:
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    process = subprocess.Popen(
        command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")


sh("nvidia-smi -L")

## 2. Save results to Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/medvisionai"
os.makedirs(DRIVE_OUT, exist_ok=True)

## 3. Get the code and install dependencies

The repository pins TensorFlow; the CUDA libraries for that exact version are added on top.

In [ ]:
sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
# Colab points Matplotlib at its notebook backend, which does not exist in the
# project's own environment; use the headless backend for saved figures.
os.environ["MPLBACKEND"] = "agg"

REPO = "/content/MedVisionAI"
# Leave the repository before deleting it, so this cell can be re-run safely.
os.chdir("/content")
sh(f"rm -rf {REPO} && git clone --depth 1 https://github.com/OmkarJayvantJadhav/MedVisionAI.git {REPO}")
os.chdir(f"{REPO}/ml")
sh("uv sync --locked")

# Colab sets UV_SYSTEM_PYTHON, which points `uv pip` at the system Python
# (with its own TensorFlow). Always name the project's interpreter explicitly.
VENV_PYTHON = f"{REPO}/ml/.venv/bin/python"
# Read the locked version from package metadata; importing TensorFlow here is slow.
tf_version = subprocess.run(
    [VENV_PYTHON, "-c", "import importlib.metadata as m; print(m.version('tensorflow'))"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f"Installing the CUDA build of TensorFlow {tf_version} into the project environment (a few minutes)...")
sh(f'uv pip install --python {VENV_PYTHON} "tensorflow[and-cuda]=={tf_version}"')
# Keep the CUDA packages: stop later `uv run` calls from re-syncing them away.
os.environ["UV_NO_SYNC"] = "1"
sh("uv run python -c \"import tensorflow as tf; gpus = tf.config.list_physical_devices('GPU'); print('TensorFlow', tf.__version__, 'GPUs:', gpus); assert gpus\"")

## 4. Download ISIC 2019

In [ ]:
BASE = "https://isic-challenge-data.s3.amazonaws.com/2019"
ISIC = "/content/isic"
os.makedirs(ISIC, exist_ok=True)
for name in ["ISIC_2019_Training_GroundTruth.csv", "ISIC_2019_Training_Metadata.csv", "ISIC_2019_Training_Input.zip"]:
    # Download to a .part file and rename when complete, so re-runs skip finished files.
    sh(f"cd {ISIC} && [ -f {name} ] || (curl -sS --retry 5 -C - -o {name}.part {BASE}/{name} && mv {name}.part {name})")
print("Unzipping 25,331 images (a few minutes)...")
sh(f"cd {ISIC} && unzip -qn ISIC_2019_Training_Input.zip && ls ISIC_2019_Training_Input | wc -l")

## 5. Prepare lesion-grouped splits and check for leakage

The notebook stops here if any lesion or duplicate image crosses splits.

In [ ]:
sh("rm -rf data/isic2019_raw")  # prepare-isic2019 needs an empty destination
sh(
    "uv run medvision-ml prepare-isic2019"
    f" --images {ISIC}/ISIC_2019_Training_Input"
    f" --ground-truth {ISIC}/ISIC_2019_Training_GroundTruth.csv"
    f" --metadata {ISIC}/ISIC_2019_Training_Metadata.csv"
    " --dest data/isic2019_raw"
)
sh("uv run medvision-ml split --source data/isic2019_raw --overwrite")
sh("uv run medvision-ml check-leakage")
sh(f'mkdir -p "{DRIVE_OUT}/reports" && cp outputs/leakage_report.json "{DRIVE_OUT}/reports/"')

## 6. Train

Decoded images are cached on local disk (about 12 GB) because they do not fit in RAM.

In [ ]:
RUN = "isic2019-effb0"
os.environ["MEDVISION__DATA__CACHE_DIR"] = "/content/tf_cache"
print("Training: progress is printed at the end of each epoch; this takes hours.")
sh(f"uv run medvision-ml train --run-name {RUN}")
sh(f'mkdir -p "{DRIVE_OUT}/runs" && cp -r outputs/runs/{RUN} "{DRIVE_OUT}/runs/" && cp -r artifacts/{RUN} "{DRIVE_OUT}/runs/{RUN}/model"')

## 7. Evaluate on the held-out test split

In [ ]:
sh(f"uv run medvision-ml evaluate --model artifacts/{RUN}/model.keras")
sh(f"cat outputs/evaluation/{RUN}/test/classification_report.txt")
sh(f'cp -r outputs/evaluation/{RUN} "{DRIVE_OUT}/reports/evaluation"')

## 8. Export the serving artifacts

In [ ]:
VERSION = "skin-effb0-v2.0.0"
sh(f"uv run medvision-ml export --model artifacts/{RUN}/model.keras --version {VERSION} --output export/{VERSION}")
sh(f'cp -r export/{VERSION} "{DRIVE_OUT}/" && cd export && zip -qr "{DRIVE_OUT}/{VERSION}.zip" {VERSION}')
print(f"Done. Everything is in Google Drive: MyDrive/medvisionai/ ({VERSION}.zip has the model files).")

## Next steps

- Share `reports/evaluation/test/metrics.json` and `classification_report.txt`
  so the README and model card can report real numbers.
- The model files in `skin-effb0-v2.0.0.zip` become the `model-v2.0.0` release,
  published under CC BY-NC 4.0 with the attribution in `docs/DATASETS.md`.